# Systemy rekomendacyjne

## Wst?p

Celem laboratorium jest poznanie podstaw system?w rekomendacyjnych. Zapoznasz si? na nim z nast?puj?cymi tematami:
* budow? system?w rekomendacyjnych typu collaborative filtering (CF), w szczeg?lno?ci z:
  * macierz? interakcji u?ytkownik-przedmiot (user-item matrix)
  * poj?ciem biasu u?ytkownika i przedmiotu
  * analiz? zbior?w danych do CF
  * metrykami jako?ci dla system?w rekomendacyjnych
* algorytmami globalnej rekomendacji:
  * metodami podstawowymi (baselines)
  * metodami bayesowskimi (Bayesian average)
* algorytmami personalizowanej rekomendacji typu CF, w szczeg?lno?ci z:
  * najbli?szych s?siad?w (neighborhood-based) typu user-based oraz item-based
  * rozk?adem macierzowym (matrix factorization) typu MF oraz FunkSVD

Jak zwykle, mo?esz albo korzysta? z Google Colab, albo z w?asnego komputera. W obu przypadkach trzeba doinstalowa? troch? bibliotek.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/apohllo/sztuczna-inteligencja/blob/master/lab7/lab_7.ipynb)

## Kr?tki wst?p teoretyczny

**Systemy rekomendacyjne (recommender systems)** to dowolne metody maj?ce rekomendowa? u?ytkownikom (users) pewne przedmioty (items). Korzysta z nich praktycznie ka?da wi?ksza firma: Netflix (filmy - "Top picks for you"), Spotify (muzyka, "Recommended for playlist"), Amazon (sklep - "frequently bought together) etc. Maj? niesamowicie praktyczne zastosowanie i s? jednym z najwcze?niej oraz najcz??ciej wdra?anych metod uczenia maszynowego.

Jest to bardzo szeroka dziedzina, o bardzo r??norodnych podej?ciach. W szczeg?lno?ci mo?na wyr??ni? grupy:
1. **Collaborative filtering (CF)** - oparte o histori? interakcji u?ytkownik?w z przedmiotami, czyli zwykle o histori? ocen. St?d pochodz? np. rekomendacje "u?ytkownicy podobni do ciebie ogl?dali tak?e X", gdzie podobie?stwo mierzy si? na podstawie tego, jak bardzo podobne mieli?my w przesz?o?ci oceny do innych u?ytkownik?w. Co wa?ne, takie podej?cie nie wymaga ?adnej in?ynierii cech, a jedynie zapami?tania historii ocen / transakcji / interakcji!
2. **Content-based (CB)** - du?o bardziej podobne do klasycznego ML, tworzymy wektory cech dla przedmiot?w, u?ytkownik?w i wykorzystujemy je w klasyfikacji (np. rekomendowa? lub nie) lub regresji (np. liczba gwiazdek).
3. **Algorytmy hybrydowe** - ??cz?ce podej?cia CF i CB podczas nauki. S? zazwyczaj bardziej z?o?one i wymagaj? odpowiednio du?ych zbior?w danych.

Dodatkowo mo?emy podzieli? problemy rekomendacji na dwa rodzaje, w zale?no?ci od tego, czym s? nasze **oceny (ratings)**:
1. **Explicit feedback** - kiedy u?ytkownicy jawnie podaj? oceny, np. ocena hotelu w skali 1-10, liczba gwiadek dla przedmiotu. Wymaga to wi?kszej proaktywno?ci u?ytkownik?w, wi?c potencjalnie mo?emy mie? mniej danych, ale s? cz?sto bardziej precyzyjne. S? te? typowo prostsze teoretycznie (matematycznie), bo maj? znany z g?ry, ograniczony zakres mo?liwych warto?ci.
2. **Implicit feedback** - kiedy jako?? przedmiotu wyznaczaj? akcje u?ytkownik?w, np. liczba klikni??, liczba udost?pnie?. Takie informacje mo?na gromadzi? automatycznie i bardzo ?atwo, ale mog? by? ma?o precyzyjne (np. przypadkowe klikni?cia, boty). Algorytmy dla takich problem?w s? te? ci??sze do zaprojektowania, bo mamy tylko warto?ci nieujemne i typowo nieograniczone z g?ry.

Same rekomendacje mog? by? dwojakiego rodzaju:
1. **Globalne (global)** - bior? pod uwag? og?lne cechy przedmiotu i s? oceniane dla ca?ej spo?eczno?ci, nie pod konkretnych u?ytkownik?w. Korzystaj? z nich typowo strony z wiadomo?ciami, ?eby u?o?y? kolejno?? post?w na stronie, np. HackerNews, Reddit. Przydaj? si? te?, gdy nie mamy do?? informacji o u?ytkowniku, aby dokona? personalizacji.
2. **Personalizowane (personalized)** - zasadnicze zastosowanie system?w rekomendacyjnych, w kt?rym "profilujemy" u?ytkownika lub przedmiot, tak, aby nauczy? sie relacji mi?dzy nimi i sugerowa? to, co konkretn? osob? mo?e interesowa?.

Na tym laboratorium skupimy si? na systemach typu collaborative filtering, bo s?:
1. Ciekawsze i bardziej unikatowe na tle tych algorytm?w, kt?re ju? poznali?my.
2. Cz?sto o wiele ?atwiejsze w praktycznej implementacji, gdy? nie wymagaj? feature engineeringu.
3. Bardzo szybkie i skalowalne.
4. Zazwyczaj lepsze pod wzgl?dem wynik?w od system?w content-based.

Skupimy si? na systemach typu explicit ranking, bo s? nieco prostsze i popularniejsze. Poznamy za to i systemy globalne, i personalizowane.

Czemu wi?c korzysta? z innego podej?cia ni? CF? O tym przekonasz si? w p??niejszej cz??ci labu :)

## Biblioteki do system?w rekomendacyjnych

Do system?w rekomendacyjnych nie istnieje jedna standardowa, powszechnie przyj?ta biblioteka, taka jak Scikit-learn. Jest to bowiem zbyt rozleg?a dziedzina, oparta o bardzo r??norodne podej?cia i rozwi?zania, aby da?o si? j? zamkn?? w jednej bibliotece ze sp?jnym interfejsem. Mo?na jednak wyr??ni? zbi?r najpopularniejszych bibliotek. Co wa?ne, praktyczne systemy implementuje si? jednak cz?sto od zera, pod konkretny problem.

1. [Surprise](https://surpriselib.com/) - od niedawna `scikit-surprise` ze wzgl?du na implementacj? interfejs?w ze Scikit-learn'a. Implementuje algorytmy typu explicit rating collaborative filtering.
2. [Implicit](https://benfred.github.io/implicit/) - podobna do Surprise, implementuje algorytmy typu implicit rating collaborative filtering.
3. [LibRecommender](https://github.com/massquantity/LibRecommender) - rozbudowana biblioteka, implementuj?ca r??ne podej?cia: collaborative filtering, feature-based, oraz hybrydowe. Zawiera algorytmy pisane od zera, w TensorFlow (niestety v1) oraz w PyTorchu, z wielu artyku??w naukowych. Ma jednak do?? specyficzny, niekoniecznie intuicyjny interfejs.
4. [Spark MLlib](https://spark.apache.org/docs/latest/ml-collaborative-filtering.html) - de facto standard w pracy z wielkimi zbiorami danych, cz?stymi w systemach rekomendacyjnych. Implementuje explicit oraz implicit collaborative filtering.
5. [PyTorch Geometric](https://pytorch-geometric.readthedocs.io/en/latest/) - de facto standard dla grafowych sieci neuronowych (Graph Neural Networks, GNNs), kt?re s? m. in. najnowszym trendem w systemach rekomendacyjnych opartych o grafy (graph-based recommender systems).

Dodatkowo dla podej?cia content-based (opisane, ale nie implementowane w tym laboratorium) mo?na u?y? dowolnej biblioteki do uczenia nadzorowanego, typowo Scikit-learn lub Spark MLlib.

Na tym laboratorium wykorzystamy `Surprise` ze wzgl?du na prostot? u?ycia. Dodatkowo u?yjemy `recmetrics`, aby obliczy? metryki specyficzne dla system?w rekomendacyjnych, kt?rych nie implementuje Scikit-learn.

In [ ]:
!pip install scikit-surprise recmetrics

## ?ci?ganie, ?adowanie i eksploracja danych

Na pocz?tek ?ci?gniemy nasz zbi?r danych. B?dziemy pracowa? na zbiorze MovieLens-100k, czyli zbiorze 100 tysi?cy ocen film?w. Strona MovieLens udost?pnia wiele rozmiar?w tego zbioru danych, a ten b?dzie odpowiednio szybki na potrzeby edukacyjne. W praktyce wykorzystuje si? zbiory rozmiaru co najmniej takiego, jak MovieLens-1M (zbi?r miliona ocen).

Opis plik?w mo?na znale?? w [readme](https://files.grouplens.org/datasets/movielens/ml-100k-README.txt). Najwa?niejsze fragmenty:
```
u.data     -- The full u data set, 100000 ratings by 943 users on 1682 items.
              Each user has rated at least 20 movies.  Users and items are
              numbered consecutively from 1.  The data is randomly
              ordered. This is a tab separated list of
	         user id | item id | rating | timestamp.
              The time stamps are unix seconds since 1/1/1970 UTC   
```

Zbi?r co prawda ma ju? przygotowany podzia? do 5-krotnej walidacji skro?nej (pliki `u1.base`, `u1.test` etc.), ale my wykonamy ten podzia? sami. Gotowych podzia??w u?ywa si? w pracach naukowych, aby m?c por?wnywa? wyniki r??nych algorytm?w na dok?adnie tych samych zbiorach treningowych i testowych.

In [ ]:
!wget -N https://files.grouplens.org/datasets/movielens/ml-100k.zip

In [ ]:
!unzip -n ml-100k.zip

In [ ]:
import os
import pandas as pd


df = pd.read_csv(
    os.path.join("ml-100k", "u.data"),
    sep="\t",
    header=None,
    names=["user_id", "item_id", "rating", "timestamp"],
)
df.head()

In [ ]:
print(f"Number of reviews: {len(df)}")
print(f"Ratings range: {df.rating.min(), df.rating.max()}")

Tabela w formacie jak powy?ej to de facto s?ownik `(user_id, item_id) -> rating`. Jest zatem idealna do podej?cia collaborative filtering, w kt?rym dla **u?ytkownik?w (users)** mamy ich **oceny (ratings)** wybranych **przedmiot?w (items)**. Tutaj oczywi?cie przedmiotami s? filmy. Mo?na by zatem z takich danych zbudowa? **macierz ocen (ratings matrix)**, w kt?rej wiersze by?yby u?ytkownikami, kolumny przedmiotami, a kom?rki zawiera?yby oceny.


W przysz?o?ci chcemy zatem **przewidywa? warto?ci brakuj?ce** macierzy ocen. Mamy tu zatem poniek?d problem regresji - chcemy dosta? warto?? ci?g??, np. na ile u?ytkownik oceni?by film, kt?rego jeszcze nie widzia?. P??niejsza rekomendacja to po prostu wybranie najwy?szych predykcji i zaproponowanie tych w?a?nie przedmiot?w. Nazywa si? to czasem problemem **uzupe?nienia macierzy (matrix completion)**.

W praktyce **nigdy** nie nale?y budowa? macierzy ocen explicite. Zwyczajnie nie zmie?ci?aby si? ona do pami?ci dla zbior?w o prawdziwym rozmiarze, kiedy mamy setki tysi?cy u?ytkownik?w i przedmiot?w. Dodatkowo zwyczajnie nie ma to sensu, bo nasze macierze prawie zawsze s? **rzadkie (sparse)**, tzn. maj? wype?nion? tylko nieznaczn? liczb? p?l. Reszta jest nieznana - w ko?cu pojedynczy cz?owiek obejrzy tylko niewielk? cz??? wszystkich film?w z Netflixa, nie m?wi?c ju? o wystawieniu im ocen.

**Zadanie 1 (0.25 punktu)**

Oblicz g?sto?? (density) macierzy ocen dla naszego zbioru danych. Jest to liczba ocen, podzielona przez rozmiar macierzy ocen (liczba u?ytkownik?w * liczba przedmiot?w). Wynik przedstaw w procentach, zaokr?glony do 4 miejsc po przecinku. Pami?taj, ?eby uwzgl?dni? tylko unikatowych u?ytkownik?w i przedmioty.

In [ ]:
distinct_movies = df.groupby(['item_id']).size().count()
distinct_users = df.groupby(['user_id']).size().count()
distinct_pairs = df.groupby(['user_id', 'item_id']).size().count()

print(f"G?sto?? macierzy: {100 * distinct_pairs / (distinct_movies * distinct_users):.4f} %")

W praktyce cz?sto zbiory maj? g?sto?? rz?du 1% lub mniejsz?. Jest to te? pozytywne - w ko?cu to dzi?ki temu mamy komu robi? rekomendacje (i czego).

Warto zauwa?y?, ?e nasz zbi?r zawiera tylko tych u?ytkownik?w, kt?rzy ocenili przynajmniej 20 film?w, a zatem wiemy o ka?dym z nich ca?kiem sporo. Unika to problemu **zimnego startu (cold start)**, w kt?rym nic nie wiemy o nowych u?ytkownikach i/lub filmach. W prawdziwych systemach jest to jednak du?e wyzwanie. Mo?na sobie z nim radzi? na kilka sposob?w:
- rekomendowa? najpopularniejsze przedmioty
- rekomendowa? przedmioty o najwy?szych ocenach
- u?y? globalnego (niepersonalizowanego) systemu rekomendacyjnego, np. przewidywanie ?redniej dla przedmiotu
- u?ywa? systemu content-based, bo radz? sobie dobrze przy ma?ej liczbie interakcji
- poprosi? u?ytkownika przy pierwszym logowaniu o podanie pierwszych preferencji (nie zawsze mo?liwe)

W zwi?zku z problemem zimnego startu systemy rekomendacyjne zwykle s? (co najmniej) dwuetapowe i maj? osobny algorytm dla nowych u?ytkownik?w/przedmiot?w oraz osobny dla tych, o kt?rych ju? co? wiemy wi?cej i mo?emy dokonywa? personalizacji.

Zbadajmy teraz rozk?ad popularno?ci poszczeg?lnych przedmiot?w w naszym zbiorze.

**Zadanie 2 (0.25 punktu)**

Narysuj wykres popularno?ci (liczby ocen) dla poszczeg?lnych przedmiot?w. Wykorzystaj do tego funkcj? `long_tail_plot()` z biblioteki `recmetrics`.

In [ ]:
from recmetrics import long_tail_plot

long_tail_plot(df, 'item_id', 'ratings', percentage=0.5, x_labels=False)

Typowo niewielka liczba przedmiot?w odpowiada za wi?kszo?? ocen. S? to rzeczy bardzo znane i popularne, nap?dzane efektem kuli ?nie?nej. Przyk?adowo, "Titanic" ogl?da i ocenia bardzo znaczna liczba u?ytkownik?w, przez sam fakt, jak bardzo znany jest ten film. My jeste?my zwykle zainteresowani **d?ugim ogonem (long tail)** naszego rozk?adu popularno?ci, czyli zwi?kszeniem popularno?ci tych przedmiot?w, kt?re s? mniej znane, a kt?re mo?emy zaoferowa? u?ytkownikom, np. nowa muzyka do odkrycia.

**Eksploracja danych - podsumowanie**

1. W systemach typu collaborative filtering operujemy na macierzy ocen, gdzie wierszami s? u?ytkownicy, kolumnami oceny, a w kom?rkach znajduj? si? oceny.
2. Macierz ocen jest zwykle bardzo rzadka.
3. Kiedy niewiele wiemy o u?ytkowniku lub przedmiocie, to mamy problem zimnego startu, z kt?rym trzeba sobie w jaki? spos?b poradzi?.
4. Cz?sto wyst?puje zjawisko d?ugiego ogona, czyli dominacji niewielkiej grupy bardzo popularnych przedmiot?w.

## Walidacja modeli, prosty model bazowy

Na pocz?tek, zanim zaczniemy budowa? nasze modele, trzeba wyodr?bni? zbi?r testowy. Mamy tutaj kilka mo?liwo?ci. Po pierwsze, mo?na po prostu losowo, tak jak to robili?my do tej pory, i tak bardzo cz?sto si? robi.

Zbi?r testowy ma jednak symulowa? przysz?e dane, przybli?a? zdolno?? generalizacji modelu, a my mamy do dyspozycji znaczniki czasowe, z kiedy pochodz? dane oceny. Mo?na by wi?c u?y? **podzia?u czasowego (time split)**, czyli wyodr?bni? najnowsze oceny do zbioru testowego, a konkretnie najnowsze oceny per u?ytkownik. Stanowi to bardzo dobr? symulacj? tego, jak w praktyce dzia?a system.

Powy?sze podej?cia maj? jednak pewne ryzyko - mo?e si? zdarzy?, ?e tak wylosujemy zbi?r testowy, ?e dla jakiego? u?ytkownika 90% ocen jest w zbiorze testowym, wi?c spowodujemy u niego przypadkiem problem zimnego startu. Analogicznie mo?e by? przy podziale czasowym, kiedy jaki? nowy u?ytkownik by? aktywny tylko niedawno i by? mo?e nawet wszystkie jego predykcje trafi?yby do zbioru testowego. Dlatego mo?na stosowa? **podzia? per u?ytkownik**, wyodr?bniaj?c np. losowe 10% ocen ka?dego u?ytkownika jako zbi?r testowy.

Jak wida?, jest tu nieco ci??ej ni? przy zwyk?ej klasyfikacji czy regresji. Dla uproszczenia wykorzystamy zwyk?y podzia? losowy. Implementacje innych metod mo?na znale?? np. w bibliotece LibRecommender.

Surprise definiuje 2 wa?ne klasy: `Dataset` i `Trainset`. Ta pierwsza reprezentuje surowe dane, a druga wst?pnie przetworzone dane do treningu lub testowania. Interfejs jest tutaj do?? dziwny, ale w skr?cie:
- do zwyk?ych algorytm?w id? `train_set` i `test_set`
- do `GridSearchCV` id? `data_train` i `test_set`

In [ ]:
from copy import deepcopy

from sklearn.model_selection import train_test_split
from surprise.dataset import Dataset, Reader

reader = Reader(rating_scale=(df["rating"].min(), df["rating"].max()))
dataset = Dataset.load_from_df(df[["user_id", "item_id", "rating"]], reader=reader)

ratings_train, ratings_test = train_test_split(
    dataset.raw_ratings, test_size=0.2, random_state=1
)

data_train = deepcopy(dataset)
data_train.raw_ratings = ratings_train

train_set = data_train.build_full_trainset()
test_set = data_train.construct_testset(ratings_test)

Na pocz?tek zaimplementujemy model, kt?ry przewiduje po prostu warto?? ?redni? dla ka?dego przedmiotu. Nie ma on ?adnych hiperparametr?w, wi?c nawet nie b?dziemy potrzebowa? zbioru walidacyjnego. Jest to bardzo dobry **model bazowy (baseline)** w systemach rekomendacyjnych.

**Zadanie 3 (1 punkt)**

Uzupe?nij kod klasy `ItemAveragePredictor`, kt?ra przewiduje warto?? ?redni? dla ka?dego przedmiotu. Mo?e ci si? tutaj przyda? atrybut `ir` (item rating) klasy `Trainset` - [dokumentacja](https://surprise.readthedocs.io/en/stable/trainset.html), oraz [dokumentacja tworzenia w?asnych algorytm?w](https://surprise.readthedocs.io/en/stable/building_custom_algo.html).

Dobrym pomys?em b?dzie przechowywanie danych w postaci atrybutu b?d?cego s?ownikiem w `.fit()`, ?eby zapami?ta? mapowanie `item_id` -> ?rednia ocena.

Uwaga - zgodnie z konwencj? ze Scikit-learn atrybuty, kt?rych warto?ci s? obliczane (estymowane) na podstawie danych treningowych, s? tworzone w metodzie `.fit()` i maj? underscore `_` na ko?cu nazwy, np. `self.ratings_`.

In [ ]:
import numpy as np
from surprise import AlgoBase, PredictionImpossible


class ItemAveragePredictor(AlgoBase):
    def __init__(self):
        AlgoBase.__init__(self)

    def fit(self, trainset):
        AlgoBase.fit(self, trainset)

        # mapping: item_id -> average rating
        self.ratings_ = dict(map(lambda x: (x[0], np.mean([r for user, r in x[1]])), trainset.ir.items()))

        return self

    def estimate(self, u, i):
        if not (self.trainset.knows_user(u) and self.trainset.knows_item(i)):
            raise PredictionImpossible("User and/or item is unknown.")

        return self.ratings_[i]

In [ ]:
algo = ItemAveragePredictor()
algo.fit(train_set)
pred_item_avg = algo.test(test_set)
pred_item_avg[:5]

Musimy teraz oceni? jako?? naszego algorytmu. Jako ?e mamy tu problem regresji, to naturalnym wyborem s? **RMSE (root mean squared error)** oraz **MAE (Mean Absolute Error)**. Poka?? nam one, jak bardzo ?rednio nasz model myli si? w przewidywaniu ratingu.

RMSE to po prostu pierwiastek b??du ?redniokwadratowego (MSE). Ma tak? sam? wad? przy ewaluacji jak MSE - zwraca zbyt du?? uwag? na obserwacje odstaj?ce (outliers). Dzi?ki pierwiastkowaniu ma t? sam? jednostk?, co oryginalne dane.
$$\large
RMSE = \sqrt{MSE} = \sqrt{\frac{1}{N} \sum_{i=1}^N \left( y_i - \hat{y}_i \right)^2}
$$

MAE to ?rednie odchylenie predykcji od warto?ci prawdziwej. Dzi?ki zastosowaniu warto?ci bezwzgl?dnej zamiast kwadratu jest miar? bardziej odporn? na outliery i dlatego cz?sto wykorzystywan? przy ewaluacji. Ma naturalnie t? sam? jednostk?, co mierzona warto??.
$$\large
MAE = \frac{1}{N} \sum_{i=1}^N \left| y_i - \hat{y}_i \right|
$$

Ze wzgl?du na to, ?e Surprise nie zwraca zwyk?ego wektora Numpy'a, tylko obiekty `Prediction`, trzeba u?y? metryk z tej biblioteki. Zwykle nie stanowi to problemu, a dodatkowo mamy te? do dyspozycji wszystko, co implementuje biblioteka recmetrics.

In [ ]:
from surprise.accuracy import rmse, mae

rmse(pred_item_avg, verbose=True)
mae(pred_item_avg, verbose=True);

Wygl?da na to, ?e nawet najprostszy model nie radzi sobie wcale tak ?le. Ale s? to tylko przewidywane warto?ci - zobaczmy faktyczne rekomendacje. W praktyce mamy ograniczone miejsce, np. ma?o kto popatrzy na wi?cej ni? pierwsze 5-10 rekomendowanych film?w. W zwi?zku z tym niewa?ne nawet, co b?dzie dalej - liczy si? dla nas **top k** predykcji.

Zgromadzimy teraz faktyczne najlepsze oceny ze zbioru testowego dla ka?dego u?ytkownika, rekomendacje naszego systemu i zbierzemy je w jednej DataFrame.

In [ ]:
from surprise import Prediction


def get_user_recommendations(user_rec_items: pd.Series) -> list[int]:
    return user_rec_items.sort_values(ascending=False).index.tolist()


def get_recommendations(predictions: list[Prediction]) -> pd.DataFrame:
    df_pred = pd.DataFrame(predictions)
    df_pred = df_pred.drop(columns="details")
    df_pred.columns = ["user_id", "item_id", "actual", "prediction"]

    df = (
        df_pred.groupby("user_id", as_index=False)["item_id"]
        .agg({"actual": (lambda x: list(x))})
        .set_index("user_id")
    )

    df_pivot = df_pred.pivot_table(
        index="user_id", columns="item_id", values="prediction"
    ).fillna(0)

    df["recommendations"] = [
        get_user_recommendations(df_pivot.loc[user_id]) for user_id in df.index
    ]

    return df


rec_item_avg = get_recommendations(pred_item_avg)
rec_item_avg

Teraz kiedy mo?na por?wna? faktyczne predykcje, patrz?c np. na top 5, to nie wygl?da to ju? tak dobrze, jak trzeba wybra? konkretne filmy. Do mierzenia jako?ci w?r?d top k predykcji s?u?? metryki:
- mean average precision at k (MAP@k)
- mean average recall at k (MAR@k)
- Fraction of Concordant Pairs (FCP)
- Normalized Discounted Cumulative Gain (NDCG)

S? one u?ywane w systemach rekomendacyjnych, ale te? w wyszukiwarkach i niekt?rych problemach NLP. Dla MAP i MAR dok?adny opis, krok po kroku, mo?esz znale?? [tutaj](https://sdsawtelle.github.io/blog/output/mean-average-precision-MAP-for-recommender-systems.html) i [tutaj](https://machinelearninginterview.com/topics/machine-learning/mapatk_evaluation_metric_for_ranking/). FCP doskonale opisuje [oryginalny artyku?](https://www.ijcai.org/Proceedings/13/Papers/449.pdf). [Tutaj](https://finisky.github.io/2019/04/24/ndcg/) kr?tki i tre?ciwy artyku? o NDCG.

### MAP@k

Zdefiniujmy:
- True Positive (TP) - przedmiot, kt?ry naprawd? jest w top k (*relevant*) i kt?ry nasz system zarekomendowa? w top k przedmiot?w
- False Positive (FP) - przedmiot, kt?ry nie jest w top k (*nonrelevant*), ale nasz model go zarekomendowa? w top k

"Precision at k" to precyzja (precision), obliczona dla top k przedmiot?w. Oznaczmy przez $r_k$ liczb? TP (*relevant items*) w?r?d top k przedmiot?w.
$$\large
P@k = \frac{\text{number of relevant items in top }k}{k} = \frac{r_k}{k}
$$

"Average P@k" to po prostu P@k obliczone dla r??nych $i=1,2,...,k$ i u?rednione. Taka agregacja bardzo penalizuje umieszczanie nieciekawych przedmiot?w na wysokich miejscach, bo uwzgl?dniamy tylko precyzj? dla tych top k, gdzie prawid?owo zauwa?yli?my TP.
$$\large
AP@k = \frac{1}{r_k} \sum_{i=1}^{k} \left( P@i \text{ if i-th item is relevant} \right)
$$

MAP@k to AP@k (average precision at k), u?rednione dla wszystkich $|U|$ u?ytkownik?w:
$$\large
MAP@k = \frac{1}{|U|} \sum_{u=1}^{|U|}AP@k(u)
$$

Im ni?sze $k$, tym surowsi jeste?my i tym ni?sze b?d? wyniki - nasz algorytm ma mniej miejsca na b??d. Typowo $k=5$ lub $k=10$. Zakres warto?ci MAP@k to $[0, 1]$.

MAP@k przyk?ada bardzo du?? wag? do tego, ?eby na pierwszych miejscach trafi?y si? jak najlepsze przedmioty. Jest zatem bardzo wa?ne, kiedy mamy ma?o miejsc do dyspozycji, np. przy rekomendacji film?w na g??wnej stronie (Netflix).

### FCP

FCP (Fraction of Concordant Pairs) jest rzadziej u?ywan?, ale bardzo intuicyjn? metryk?. Ide? jest uog?lnienie metryki AUROC (ROC AUC) na algorytmy rankuj?ce, a wi?c systemy rekomendacyjne. Ma zakres warto?ci $[0, 1]$.

Liczba zgodnych par (*concordant pairs*) $n_c^u$ dla u?ytkownika $u$ to liczba par przedmiot?w, kt?re zosta?y prawid?owo uporz?dkowane przez ranker. Innymi s?owy, gdy mamy prawdziwy ranking ocen u?ytkownika oraz przewidywany, to jest to liczba par przedmiot?w, kt?re prawid?owo u?o?yli?my (lepszy przedmiot wy?ej ni? gorszy).
$$\large
n_c(u) = |\{ (i,j) | \hat{r}_{ui} > \hat{r}_{uj} \text{ and } r_{ui} > r_{uj}\}|
$$

Pary niezgodne (*discordant pairs*) liczy si? podobnie:
$$\large
n_d(u) = |\{ (i,j) | \hat{r}_{ui} > \hat{r}_{uj} \text{ and } r_{ui} \leq r_{uj}\}|
$$

Proporcja par zgodnych do wszystkich, zsumowana dla wszystkich u?ytkownik?w, to FCP:
$$\large
FCP = \frac{n_c}{n_c + n_c} = \frac{\sum_{i=1}^n n_c(u)}{\sum_{i=1}^n n_c(u) + n_d(u)}
$$

Mo?na tak?e obliczy? FCP@k, ograniczaj?c si? do pierwszych k predykcji.

Metryka FCP przyk?ada mniejsz? wag? ni? MAP@k do tego, ?eby najlepsze przedmioty by?y jak najwy?ej. Skupia si? natomiast na tym, ?eby lepsze przedmioty by?y powy?ej gorszych. Dzia?a wi?c lepiej dla rekomendacji d?u?szych list, kiedy pierwsze pozycje nie s? a? tak wa?ne, np. przy rekomendowaniu playlist muzyki (Spotify).

**Zadanie 4 (1 punkt)**

Uzupe?nij kod funkcji `ap_k`, kt?ra oblicza AP@k dla pojedynczego u?ytkownika. Pami?taj, aby ograniczy? si? do najwy?szych (pierwszych) `k` przedmiot?w dla rekomendacji oraz predykcji. W przypadku, gdy model nie mia? ?adnej dobrej predykcji ($r_k = 0$), zwr?? 0.

Nast?pnie oblicz i wypisz MAP@k oraz FCP (k=10) dla naszego modelu ?redniej przedmiotu. Warto?ci podaj w procentach.

In [ ]:
def ap_k(y_true: list[int], y_pred: list[int], k: int) -> float:
    relevant_items = [0 for _ in range(k + 1)]
    for i in range(1, k + 1):
        relevant_items[i] = relevant_items[i-1] + (y_pred[i-1] in y_true[:i])

    if relevant_items[-1] == 0:
        return 0

    ap_at_k = 0
    for i in range(1, k + 1):
        if y_pred[i-1] in y_true:
            ap_at_k += relevant_items[i] / i

    ap_at_k /= relevant_items[-1]
    return ap_at_k


In [ ]:
from collections import defaultdict
from operator import itemgetter

from surprise.accuracy import fcp


def map_k(df: pd.DataFrame, k: int) -> float:
    ap_k_values = []
    for idx, row in df.iterrows():
        actual, recommendations = row
        ap_k_val = ap_k(actual, recommendations, k)
        ap_k_values.append(ap_k_val)

    return np.mean(ap_k_values)


def fcp_k(predictions: list[Prediction], k: int) -> float:
    top_k = defaultdict(list)
    for uid, iid, true_r, est, _ in predictions:
        top_k[uid].append((iid, est))

    user_item_id_pairs = set()

    for user_id, user_ratings in top_k.items():
        user_ratings.sort(key=itemgetter(1), reverse=True)
        for item_id, rating in user_ratings[:k]:
            user_item_id_pairs.add((user_id, item_id))

    predictions_top_k = [
        pred for pred in predictions if (pred[0], pred[1]) in user_item_id_pairs
    ]

    return fcp(predictions_top_k, verbose=False)

In [ ]:
map_k_item_avg = map_k(rec_item_avg, k=10)
fcp_item_avg = fcp_k(pred_item_avg, k=10)

print(f"Item average model MAP@k (k=10): {100 * map_k_item_avg:.2f}%")
print(f"Item average model FCP@k (k=10): {100 * fcp_item_avg:.2f}%")

Zobaczymy, ?e ten wynik da si? jeszcze poprawi?.

Metryki MAP@k i MAR@k maj? jednak pewn? wad? - preferuj? sugerowanie popularnych tre?ci przez model, bo mo?na je ?atwo umie?ci? wysoko w rekomendacji i ?atwo podbi? sobie precyzj?. W ten spos?b rekomendacje by?yby s?abo personalizowane. Dlatego wykorzystuje si? szereg innych metryk, g??wnie bior?cych pod uwag? r??norodno?? i personalizacj? rekomendacji, na przyk?ad:
- pokrycie (*coverage*) - procent przedmiot?w ze zbioru, kt?ry nasz system w og?le rekomenduje
- nowo?? (*novelty*) - zdolno?? systemu do rekomendacji zaskakuj?cych, nowych dla u?ytkownika przedmiot?w
- personalizacja (*personalization*) - miara r??nicy mi?dzy rekomendacjami dla poszczeg?lnych u?ytkownik?w

Mo?esz o nich poczyta? wi?cej na stronie [recmetrics](https://github.com/statisticianinstilettos/recmetrics) oraz w [tym artykule](https://towardsdatascience.com/evaluation-metrics-for-recommender-systems-df56c6611093).

**Pomiar jako?ci system?w rekomendacyjnych - podsumowanie**

1. Poza stosowaniem zwyk?ego podzia?u losowego train-test mo?na te? stosowa? podzia? czasowy lub per u?ytkownik.
2. Jednym z najprostszych modeli i dobrym punktem odniesienia (baseline) jest przewidywanie ?redniej per przedmiot.
3. Podstawowymi metrykami jako?ci s? metryki dla regresji: RMSE i MAE, oraz rankowania: MAP@k, MAR@k, NDCG.
4. Inne metryki, specyficzne dla rekomendacji, bior? pod uwag? jako?? personalizowanych rekomendacji, np. pokrycie, nowo??, personalizacja.

## Model ?redniej bayesowskiej

Masz ochot? na dobr? pizz? i szukasz opinii na Google Maps. Masz do wyboru 2 lokale: jeden ze ?redni? 5.0 i drugi ze ?redni? 4.8. Zauwa?asz jednak, ?e pierwszy ma tylko 5 opinii, a drugi 200. Kt?ry wybierzesz? S? spore szanse, ?e ten drugi, bo mamy wi?ksz? **pewno?? (confidence)** co do oceny takiego lokalu.

Prosty model, taki jak ?rednia przedmiotu, ma ten sam problem, co powy?ej. Sformalizowaniem idei "chc? by? pewny, ?e ocena przedmiotu jest wysoka" jest model **?redniej bayesowskiej (Bayesian average)**. Mo?liwych sformu?owa? bayesowskich jest du?o, ale og?lna idea jest zawsze taka, aby wzi?? pod uwag? rozk?ad ocen przedmiotu oraz ich liczb?. Co wa?ne, to dalej s? rekomendacje globalne - mamy jedn? predykcj? per przedmiot.

Czemu ?rednia "bayesowska"? Przypomnijmy sobie twierdzenie Bayesa:
$$\large
P(Y|X) = \frac{P(X|Y) \cdot P(Y)}{P(X)}
$$

W naszym wypadku:
1. $X$ - zbi?r danych, kt?ry jest sta?y.
2. $Y$ - przewidywane warto?ci.
3. $P(X)$ - prawdopodobie?stwo zaobserwowania naszych danych, kt?re co prawda ci??ko jest zmierzy?, ale na szcz??cie w ML zwykle mo?emy zignorowa? mianownik, bo to tylko sta?a.
4. $P(Y)$ - *prior* (*prior distribution*), czyli z g?ry za?o?ony rozk?ad prawdopodobie?stw warto?ci, kt?re przewidujemy. Cz?sto zaczynamy bez ?adnej wiedzy, wi?c zak?adamy rozk?ad jednostajny lub normalny.
5. $P(X|Y)$ - *likelihood*, wiarygodno??, czyli jak dobrze model odwzorowuje dotychczas zaobserwowane dane.
6. $P(Y|X)$ - *posterior* (*posterior distribution*), czyli docelowy rozk?ad warto?ci przewidywanych, obliczony na podstawie danych.

W kontek?cie system?w rekomendacyjnych:
- $P(Y)$ (prior) to za?o?ony z g?ry rozk?ad ocen, typowo jednostajny, czyli jest taka sama szansa na ka?d? ocen?
- $P(X|Y)$ (likelihood) to miara, jak dobrze nasz model odwzorowuje macierz ocen; jakby?my potraktowali go jako skrzynk? generuj?c? oceny, to wiarygodno?? mierzy, jak bliskie s? te generowane warto?ci wobec prawdziwych ze zbioru danych
- $P(Y|X)$ (posterior) to rozk?ad przewidywanych ocen dla poszczeg?lnych przedmiot?w

Jak wida?, dostajemy rozk?ad w wyniku. Jak dosta? konkretn? predykcj?, czyli np. liczb? gwiazdek? U?ywamy **maximum a posteriori (MAP)**, czyli bierzemy po prostu t? ocen?, dla kt?rej rozk?ad posterior ma najwi?ksz? warto??.

Wykorzystamy podej?cie opisane krok po kroku [w tym artykule](https://fulmicoton.com/posts/bayesian_rating/) oraz [tym tutorialu](https://www.algolia.com/doc/guides/managing-results/must-do/custom-ranking/how-to/bayesian-average/), w kt?rym przewidywana ocena dla $i$-tego przedmiotu (po przekszta?ceniach) to:
$$\large
r_i = \frac{C \cdot m + \text{suma ocen dla przedmiotu } i}{C + \text{liczba ocen}}
$$

gdzie:
- $m$ - prior, globalna ?rednia ocen dla wszystkich przedmiot?w
- $C$ - confidence, liczba ocen dla przedmiotu
- liczba ocen

Dodatkowe ?r?d?a:
- [artyku? o twierdzeniu Bayesa](https://towardsdatascience.com/understand-bayes-rule-likelihood-prior-and-posterior-34eae0f378c5)
- [proste i przyjazne sformu?owanie ?redniej bayesowskiej](https://arpitbhayani.me/blogs/bayesian-average)
- [bardziej wyrafinowane podej?cie oparte o doln? granic? b??du](https://www.evanmiller.org/how-not-to-sort-by-average-rating.html)
- [bardzo wyrafinowane podej?cie oparte o doln? granic? b??du - dla odwa?nych](https://www.evanmiller.org/ranking-items-with-star-ratings.html)
- [?rednia bayesowska dla danych zmiennych w czasie](https://www.evanmiller.org/bayesian-average-ratings.html)

**Zadanie 5 (1 punkt)**

Uzupe?nij kod klasy `BayesianAveragePredictor`. W metodzie `.fit()` musisz obliczy? parametry:
- sum? ocen dla ka?dego przedmiotu
- liczb? ocen dla ka?dego przedmiotu
- globaln? pewno?? (confidence, $C$)

Pewno?? oblicz jako dolny kwartyl (25 percentyl) rozk?adu liczby ocen przedmiot?w, zgodnie z [tym tutorialem](https://www.algolia.com/doc/guides/managing-results/must-do/custom-ranking/how-to/bayesian-average/#how-to-calculate-the-bayesian-average). Przyda ci si? funkcja `np.quantile()`.

Sugerowane jest u?ywanie s?ownik?w w `.fit()`, ?eby mapowa? `item_id` na odpowiedni? warto??.

W metodzie `.estimate()` musisz zastosowa? obliczone parametry we wzorze podanym powy?ej.

Dokonaj predykcji i oblicz metryki za pomoc? podanej funkcji. Skomentuj wynik w por?wnaniu do przewidywania ?redniej przedmiotu.

In [ ]:
class BayesianAveragePredictor(AlgoBase):
    def __init__(self):
        AlgoBase.__init__(self)

    def fit(self, trainset):
        AlgoBase.fit(self, trainset)

        self.global_avg_ = trainset.global_mean

        # mapping: item_id -> sum of ratings
        self.sum_ratings_ = defaultdict(lambda: 0)
        # mapping: item_id -> number of ratings
        self.count_ratings_ = defaultdict(lambda: 0)

        for user, item, rating in trainset.all_ratings():
            self.sum_ratings_[item] += rating
            self.count_ratings_[item] += 1

        distribution =  np.array(list(rating for (_, _, rating) in trainset.all_ratings()))
        # confidence (C)
        self.C_ = np.quantile(distribution, 0.25)
        return self

    def estimate(self, u, i):
        if not (self.trainset.knows_user(u) and self.trainset.knows_item(i)):
            raise PredictionImpossible("User and/or item is unknown.")

        C = self.C_
        m = self.global_avg_
        item_sum = self.sum_ratings_[i]
        item_count = self.count_ratings_[i]

        score = (C*m + item_sum) / (C + item_count)
        return score


In [ ]:
def print_metrics(
    predictions: list[Prediction], recommendations: pd.DataFrame, k: int = 10
) -> None:
    rmse(predictions, verbose=True)
    mae(predictions, verbose=True)
    map_k_value = map_k(recommendations, k=k)
    fcp_k_value = fcp_k(predictions, k=k)

    print(f"MAP@k ({k=}): {100 * map_k_value:.2f}%")
    print(f"FCP@k ({k=}): {100 * fcp_k_value:.2f}%")

In [ ]:
algo = BayesianAveragePredictor()
algo.fit(train_set)
pred_bayes_avg = algo.test(test_set)
rec_bayes_avg = get_recommendations(pred_bayes_avg)

# calculate and print metrics

print_metrics(pred_bayes_avg, rec_bayes_avg)

Co ciekawe wyniki wysz?y praktycznie takie same jak dla zwyk?ej ?redniej.

## Metody oparte o s?siedztwo

Maj?c solidne punkty odniesienia w postaci rekomendacji globalnych, mo?emy przej?? do rekomendacji personalizowanych. W metodach **opartych o s?siedztwo (neighborhood-based)** znajdujemy podobnych u?ytkownik?w do nas, albo przedmioty podobne do tych kt?re lubili?my, i na podstawie tego dokonujemy rekomendacji.

Podej?cie to jest u?ywane tak?e w innych obszarach uczenia maszynowego, np. w algorytmie k najbli?szych s?siad?w (*k nearest neighbors*, kNN), SMOTE, albo w identyfikacji os?b (znajdujemy 1 najbli?szego s?siada dla embeddingu twarzy). Wymaga ono odpowiedniej metryki, kt?ra zmierzy odleg?o?? mi?dzy wektorami, znajduj?c k najbli?szych s?siad?w, z kt?rych nast?pnie wyci?gamy informacje.

### User-based neighborhood-based CF

Idea podej?cia **user-based** jest bardzo prosta - znajd?my u?ytkownik?w podobnych do nas, kt?rzy oceniali przedmioty, kt?rych my jeszcze nie widzieli?my, i zasugerujmy to, co potencjalnie najbardziej b?dzie si? nam podoba?. Realizuje podej?cie "u?ytkownicy podobni do ciebie ogl?dali tak?e...".

Algorytm user-based collaborative filtering dzia?a nast?puj?co:
1. Dla ka?dego u?ytkownika znajd? k najbli?szych s?siad?w
2. Predykcja dla przedmiotu to ?rednia ocena s?siad?w dla tego przedmiotu, kt?rzy ocenili dany przedmiot
3. Zarekomenduj te przedmioty, kt?re maj? najwy?sz? przewidywan? ocen?

Co wa?ne, przy obliczaniu najbli?szych u?ytkownik?w bierzemy tylko te przedmioty, kt?re obaj ocenili. Przyk?adowo, je?eli u?ytkownik $u_1$ oceni? przedmioty $[1, 2, 3]$, a u?ytkownik $u_2$ oceni? przedmioty $[2, 3, 4]$, to na potrzeby obliczania ich podobie?stwa bierzemy pod uwag? tylko $[2, 3]$. Przy obliczaniu predykcji dla $i$-tego przedmiotu tak?e bierzemy pod uwag? tylko tych najbli?szych s?siad?w, kt?rzy wystawili mu ocen?.

Predykcja dla u?ytkownika $u$ i przedmiotu $i$ to:
$$\large
\hat{r}_{ui} = \frac{\sum_{v \in N_i^k(u)} \text{sim}(u, v) * r_{vi}}{\sum_{v \in N_i^k(u)} \text{sim}(u, v)}
$$
gdzie:
- $N_i^k(u)$ - $k$ najbli?szych s?siad?w dla u?ytkownika $u$, kt?rzy ocenili przedmiot $i$
- $r_{vi}$ - ocena przedmiotu $i$ przez u?ytkownika $v$
- $\text{sim}(u, v)$ - podobie?stwo u?ytkownik?w $u$ i $v$ wed?ug metryki $\text{sim}$

Co wa?ne, tutaj metryka jest podobie?stwem, tzn. wi?ksza warto?? = bardziej podobni u?ytkownicy. Typowo u?ywa si? **korelacji Pearsona (Pearson correlation)**, kt?ra przyjmuje warto?ci z zakresu $[-1, 1]$. Dzi?ki temu wiemy, kt?rzy u?ytkownicy s? bardzo podobni (blisko 1), kt?rzy maj? wr?cz przeciwny gust do naszego (blisko -1), a kt?rzy s? w og?le inni od nas (blisko 0). Niekt?re implementacje (np. Surprise) bior? pod uwag? tylko s?siad?w o nieujemnej korelacji, a inne wykorzystuj? t? informacj? z ujemn? wag?.

In [ ]:
from surprise.prediction_algorithms.knns import KNNBasic

knn_basic = KNNBasic(sim_options={"name": "pearson"})
knn_basic.fit(train_set)
pred_knn_basic = knn_basic.test(test_set)
rec_knn_basic = get_recommendations(pred_knn_basic)

print_metrics(pred_knn_basic, rec_knn_basic)

Wynik nie jest mo?e idealny, ale nie przeprowadzili?my jeszcze ?adnego tuningu hiperparametr?w.

Najwa?niejszym hiperparametrem jest **liczba s?siad?w `k`**. Trzeba wzi?? pod uwag?, ?e nie wszystkie przedmioty b?d? si? pokrywa? mi?dzy u?ytkownikami, wi?c typowo bierze si? ich do?? du?o. Je?eli dana implementacja uwzgl?dnia tylko nieujemne korelacje, to jeszcze wi?cej s?siad?w mo?e odpa??, wi?c trzeba wzi?? wi?ksz? warto??. Jest to wi?c de facto maksymalna liczba s?siad?w do uwzgl?dnienia. Im wi?ksza warto??, tym mocniejsza regularyzacja, bo u?redniamy wi?cej u?ytkownik?w. Przede wszystkim nale?y jednak wzi?? pod uwag? wielko?? naszego zbioru, szczeg?lnie liczb? u?ytkownik?w oraz g?sto??.

Drugim hiperparametrem jest **minimalna liczba s?siad?w `min_k`**. Je?eli spo?r?d `k` najbli?szych s?siad?w mniej ni? `min_k` oceni?o dany przedmiot, to mamy zimny start. Zwykle wykorzystuje si? wtedy algorytm globalny, np. przewiduj?c globaln? ?redni?. Jak wida?, system rekomendacyjny sk?ada si? w ?rodku z bardzo wielu system?w rekomendacyjnych :)

**Zadanie 6 (1 punkt)**

Przeprowad? tuning hiperparametr?w, u?ywaj?c 10-krotnej walidacji skro?nej i optymalizuj?c MAE. Jako ?e nasz zbi?r jest do?? ma?y, to sprawdzimy zakres:
```
param_grid = {
    "k": list(range(10, 51, 10)),
    "min_k": list(range(1, 4)),
    "sim_options": {"name": ["pearson"]},
    "random_state": [0],
    "verbose": [False]
}
```

Jako ?e interesuj? nas przede wszystkim same rekomendacje, optymalizuj metryk? FCP. Wypisz znalezione najlepsze hiperparametry oraz metryki na zbiorze testowym dla najlepszego modelu.

Wskaz?wki:
- `GridSearchCV` z biblioteki Surprise
- argument `refit` ma domy?ln? warto?? `False`, inaczej ni? w Scikit-learn'ie
- argument `n_jobs`
- niestety `random_state` trzeba przekaza? jako hiperparametr, API Surprise jest tutaj niezbyt dobrze zrobione ([Github issue](https://github.com/NicolasHug/Surprise/issues/212))
- analogicznie do powy?szego trzeba przekaza? do?? dziwnie `sim_options`
- analogicznie do powy?szego dzia?a przekazywanie `verbose` (?eby unikn?? zalewu tekstu)

Skomentuj wyniki i zmiany w poszczeg?lnych metrykach.

In [ ]:
from surprise.model_selection import GridSearchCV

param_grid = {
    "k": list(range(10, 51, 10)),
    "min_k": list(range(1, 4)),
    "sim_options": {"name": ["pearson"]},
    "random_state": [0],
    "verbose": [True]
}

model_grid = GridSearchCV(algo_class=KNNBasic, param_grid=param_grid, measures=['fcp'], cv=10, n_jobs=-1, refit=True, joblib_verbose=2)
model_grid.fit(data_train)

In [ ]:
print(f"Best params {model_grid.best_params['fcp']}")
model = model_grid.best_estimator['fcp']
pred_knn_basic = model.test(test_set)
rec_knn_basic = get_recommendations(pred_knn_basic)

print_metrics(pred_knn_basic, rec_knn_basic)

Jak wida? wyniki po raz kolejny s? bardzo podobne do poprzednich

Ten algorytm nie bierze jednak psychologicznych r??nic mi?dzy u?ytkownikami. Niekt?rzy u?ytkownicy b?d? ?rednio zawy?a? oceny, bo film to dla nich lu?na rozrywka, a powa?ni koneserzy mog? dawa? filmom ?rednio do?? niskie oceny. Taka tendencja to **user bias**, ale na szcz??cie mo?na go policzy? - to po prostu ?rednia ocena wystawiana przez u?ytkownika, a wi?c ?rednia z ka?dego wiersza w macierzy ocen.

Je?eli od ka?dego wiersza odejmiemy jego ?redni?, to dostaniemy **ratings deviations**, czyli nie mamy ju? w macierzy samych ocen, tylko jak bardzo ocena danego przedmiotu przez u?ytkownika r??ni si? od jego ?redniej predykcji. Taka operacja to **centrowanie (centering)**. Na takich warto?ciach mo?na te? zwyczajnie liczy? najbli?szych s?siad?w, a korelacja Pearsona dalej dzia?a dla takich danych. ?eby dokona? predykcji, przewidujemy odchylenie dla przedmiotu, a nast?pnie dodajemy je dla ?redniej danego u?ytkownika.

Mamy zatem:
$$\large
\hat{r}_{ui} = \mu_i + \frac{\sum_{v \in N_i^k(u)} \text{sim}(u, v) * (r_{vi} - \mu_v)}{\sum_{v \in N_i^k(u)} \text{sim}(u, v)}
$$

**Zadanie 7 (0.5 punktu)**

Analogicznie do poprzedniego zadania wytrenuj, zoptymalizuj i sprawd? na zbiorze treningowym user-based CF z centrowaniem (`KNNWithMeans`). Wypisz tak?e optymalny zestaw hiperparametr?w dla obu algorytm?w.

Skomentuj uzyskane hiperparametry i wyniki.

In [ ]:
from surprise.prediction_algorithms.knns import KNNWithMeans

model_grid = GridSearchCV(algo_class=KNNWithMeans, param_grid=param_grid, measures=['fcp'], cv=10, n_jobs=-1, refit=True, joblib_verbose=2)
model_grid.fit(data_train)

In [ ]:
print(f"Best params {model_grid.best_params['fcp']}")
model = model_grid.best_estimator['fcp']
pred_knn_basic = model.test(test_set)
rec_knn_basic = get_recommendations(pred_knn_basic)

print_metrics(pred_knn_basic, rec_knn_basic)

Warto?ci metryk RMSE i MAE okaza?y si? by? lepsze, lecz MAP i FCP s? raczej podobne.

### Item-based neighborhood-based CF

Idea podej?cia **item-based** jest bardzo podobna do user-based, ale znajdujemy podobne przedmioty, a nie u?ytkownik?w. Operujemy zatem na kolumnach macierzy ocen. Realizuje to podej?cie "mog? ci? jeszcze zainteresowa? przedmioty..." oraz "skoro ogl?da?e? X, to mog? spodoba? ci si?...".

Predykcja dla u?ytkownika $u$ i przedmiotu $i$ to:
$$\large
\hat{r}_{ui} = \frac{\sum_{j \in N_u^k(i)} \text{sim}(u, v) * r_{uj}}{\sum_{j \in N_u^k(i)} \text{sim}(u, v)}
$$

Podobie?stwo przedmiot?w liczymy tutaj wed?ug kolumn macierzy, a metryk? jest zwykle **podobie?stwo cosinusowe (cosine similarity)**. Wykorzystuje si? tak?e centrowanie, eliminuj?c **item bias** - przyk?adowo, "Titanic" b?dzie mia? zwykle zawy?one oceny, bo ka?dy s?ysza?, ?e to znany i dobry film, wi?c pod?wiadomie zawy?ymy mu ocen?. Metryk? po centralizacji nazywa si? czasem *adjusted cosine similarity*.

Podej?cie item-based zazwyczaj daje wi?ksz? dok?adno?? ni? used-based, tzn. ni?sze RMSE i MAE. Skutkuje to jednak ni?szym pokryciem czy nowo?ci?. Takie podej?cie potrafi by? te? bardziej czu?e na zimny start.

**Zadanie 8 (0.5 punktu)**

Zaimplementuj podej?cie item-based z metryk? cosinusow? w wariantach:
- bez normalizacji
- z centrowaniem (adjusted cosine)

Analogicznie do poprzedniego ?wiczenia zastosuj optymalizacj? hiperparametr?w, podaj najlepszy zestaw oraz wypisz metryki na zbiorze testowym.

?eby zamieni? algorytm user-based na item-based, oraz zmieni? metryk?, przyda ci si? [ten tutorial](https://surprise.readthedocs.io/en/stable/getting_started.html#tune-algorithm-parameters-with-gridsearchcv).

Skomentuj, jak? uzyskano r??nic? wzgl?dem user-based i kt?re rozwi?zanie twoim zdaniem jest lepsze.

In [ ]:
param_grid = {
    "k": list(range(10, 51, 10)),
    "min_k": list(range(1, 4)),
    "sim_options": {"name": ["cosine"],
                    'user_based': [False]},
    "random_state": [0],
    "verbose": [True]
}

model_grid = GridSearchCV(algo_class=KNNBasic, param_grid=param_grid, measures=['fcp'], cv=10, n_jobs=-1, refit=True, joblib_verbose=3)
model_grid.fit(data_train)

In [ ]:
print(f"Best params {model_grid.best_params['fcp']}")
model = model_grid.best_estimator['fcp']
pred_knn_basic = model.test(test_set)
rec_knn_basic = get_recommendations(pred_knn_basic)

print_metrics(pred_knn_basic, rec_knn_basic)

In [ ]:
model_grid = GridSearchCV(algo_class=KNNWithMeans, param_grid=param_grid, measures=['fcp'], cv=10, n_jobs=-1, refit=True, joblib_verbose=2)
model_grid.fit(data_train)

In [ ]:
print(f"Best params {model_grid.best_params['fcp']}")
model = model_grid.best_estimator['fcp']
pred_knn_basic = model.test(test_set)
rec_knn_basic = get_recommendations(pred_knn_basic)

print_metrics(pred_knn_basic, rec_knn_basic)

Wyniki s? bardzo podobne jak do we wszystkich innych modelach. Wed?ug mnie lepszym podej?ciem jest item based, gdy? znajdowanie podobnych od siebie uzytkownik?w mo?e by? trudne i przez to du?a liczba informacji nie jest u?ywana. Zazwyczaj te? liczba u?ytkownik?w jest wi?ksza ni? liczba proponowanych produkt?w, a wtedy metody item-based lepiej si? skaluj?.

### Metody oparte o s?siedztwo - podsumowanie

Podsumowanie:
1. Mo?emy wyr??ni? dwa sposoby liczenia s?siad?w: user-based (inni u?ytkownicy, wiersze macierzy) oraz item-based (inne przedmioty, kolumny macierzy).
2. Podej?cie user-based wykorzystuje zwykle korelacj? Pearsona, a item-based podobie?stwo cosinusowe.
3. U?ytkownicy oraz przedmioty maj? naturalny bias (user bias, item bias), kt?ry mo?na wyeliminowa?, stosuj?c normalizacj?: centrowanie lub standaryzacj?.

Zalety:
1. Prostota
2. Interpretowalno??, szczeg?lnie dla item-based
3. Stosunkowo niewielka czu?o?? na dob?r hiperparametr?w
4. Mo?na idealnie uwsp??bie?ni? trening oraz predykcj? (embarassingly parallel)

Wady:
1. Do?? trudna implementacja, trzeba wybiera? wsp?lne przedmioty
2. Trening jest niezbyt skalowalny dla bardzo du?ych danych
3. Czu?e na zimny start
4. Bardzo niewiele implementacji wspiera dodawanie nowych u?ytkownik?w/przedmiot?w - trzeba przetrenowywa? regularnie ca?y model

## Metody oparte o rozk?ad macierzy

Podej?cie najbli?szych s?siad?w definiuje "s?siedztwo" bardzo explicite - wymaga, by u?ytkownicy ocenili dok?adnie te same filmy, aby w og?le sprawdza?, czy s? podobni. Nie wykorzystuje to niejawnych podobie?stw mi?dzy przedmiotami i filmami. Przyk?adowo, je?eli jeden lubi filmy "Szeregowiec Ryan", "Dunkierka" i "Wr?g u bram", a drugi lubi filmy "Czas apokalipsy" i "Jak rozp?ta?em drug? wojn? ?wiatow?", to s? do siebie bardzo podobni, a jednak podej?cie user-based nawet nie b?dzie w stanie tego sprawdzi?. Item-based mog?oby tu nieco pom?c, ale tam mog? si? zdarzy? analogiczne sytuacje.

Podej?cie oparte o rozk?ad macierzy, spopularyzowane w ramach konkursu Netflix Prize 2007 przez Simona Funka ([wywiad](https://www.kdd.org/exploration_files/simon-funk-explorations.pdf), [jego blog](https://sifter.org/simon/journal/20061211.html)), rozwi?zuje w?a?nie ten problem. Stanowi kamie? milowy w systemach rekomendacyjnych, gdy? jest daje bardzo dobre wyniki, doskonale uwsp??bie?nia si? i rozprasza na wiele maszyn, a do tego jest naprawd? proste. Szczeg??owy i bardzo przyst?pny opis tego podej?cia mo?na znale?? w artykule ["Matrix factorization techniques for recommender systems" Y. Koren, R. Bell, C. Volinsky](https://datajobs.com/data-science-repo/Recommender-Systems-[Netflix].pdf).

Ide? jest, aby dokona? **rozk?adu macierzy (matrix decomposition)** na macierzy ocen $R$, przybli?aj?c j? jako iloczyn dw?ch macierzy $W$ (user matrix) i $U$ (item matrix), kt?re reprezentuj? u?ytkownik?w i przedmioty po dekompozycji:
$$\large
\hat{R} = WU^T
$$


Za???my, ?e mamy $N$ u?ytkownik?w i $M$ przedmiot?w. Kszta?ty to zatem:
$$\large
\hat{R}_{N \times M} = W_{N \times K} U_{K \times N}^T
$$

Pojawi? nam si? nowy wymiar $K$ - ka?dy u?ytkownik to teraz wektor z macierzy $W$ o d?ugo?ci $K$, a ka?dy przedmiot to wektor z macierzy $U$ o d?ugo?ci $K$. Jest to **ukryta wymiarowo?? (latent dimensionality)**, stanowi?ca hiperparametr, analogiczny np. do rozmiaru warstw sieci neuronowej. Nie s? to interpretowalne cechy, ale mo?na zauwa?y? przy dobrym modelu, ?e odwzorowuj? pewne og?lne tematy w danych. Przyk?adowo, dla film?w (przedmiot?w) mog? oznacza?, jak du?o jest poszczeg?lnych temat?w w filmie, np. "romans", "komedia", "akcja". Dla u?ytkownik?w mog? oznacza?, w jak du?ym stopniu u?ytkownik interesuje si? danym tematem. Typowe warto?ci $K$ to oko?o kilkadziesi?t-kilkaset. Ze wzgl?du na wykorzystanie latent dimension takie modele nazywa si? te? **latent factor models**.

Predykcji w takim modelu dokonuje si? przez iloczyn skalarny wektora u?ytkownika (wiersz $W$) z wektorem przedmiotu (kolumna $U$):
$$\large
r_{ui} = w_u^Tu_i
$$

Dzi?ki takiemu sformu?owaniu, je?li zainteresowanie u?ytkownika tematem $k$ b?dzie du?e, a film b?dzie zawiera? du?o tematu $k$, to ich mno?enie da du?? warto??, a zatem du?? predykcj?.

Algorytm ten nazywa si? czasem niepoprawnie SVD, bo takiej nazwy u?y? Simon Funk do opisu swojego algorytmu (jego wersja ma troch? ulepsze?; b?dziemy j? nazywa? FunkSVD). Co wa?ne, nie wykorzystujemy tutaj algorytmu SVD, bo nie potrzebujemy ca?ego jego aparatu matematycznego. Zamiast tego ten algorytm to po prostu **matrix factorization (MF)**, tudzie? **Probabilistic Matrix Factorization (PMF)** ([oryginalny artyku? PMF](https://proceedings.neurips.cc/paper/2007/file/d7322ed717dedf1eb4e6e52a37ea7bcd-Paper.pdf) dowodzi, ?e to sformu?owanie jest poprawne probabilistycznie). Trenuje si? go tak?e bez SVD, zamiast tego wykorzystuj?c spadek wzd?u? gradientu lub algorytm **Alternating Least Squares (ALS)**. Ci??ko powiedzie?, kt?re podej?cie jest lepsze, patrz np. [ta dyskusja](https://stats.stackexchange.com/questions/201279/comparison-of-sgd-and-als-in-collaborative-filtering), [ten paper](http://cs229.stanford.edu/proj2014/Christopher%20Aberger,%20Recommender.pdf).Oba podej?cia bardzo dobrze opisuje [ten artyku?](https://datasciencemadesimpler.wordpress.com/tag/alternating-least-squares/), kt?ry rozwija te? bardziej formalnie, czemu ekstrakcja "temat?w" dzia?a (spoiler: MF dokonuje implicite klasteryzacji).

Niezale?nie od podej?cia, minimalizuje si? funkcj? kosztu, czyli r??nic? mi?dzy naszym przybli?eniem $\hat{R} = WU^T$ a prawdziw? macierz? $R$. Zwykle wykorzystuje si? tutaj b??d ?redniokwadratowy, w zapisie macierzowym $||\hat{R} - R||_2^2$. Zapisuj?c to r?cznie:
$$\large
L = \sum_{u, i \in \Omega} \left( r_{ui} - \hat{r}_{ui} \right)^2
$$
gdzie $\Omega$ to zbi?r wszystkich wype?nionych kom?rek w macierzy ocen.

Jako ?e mamy dwie macierze do nauczenia, $W$ oraz $U$, to mamy pochodn? po wektorach $w$ oraz po wektorach $u$. Po przekszta?ceniach dostajemy:
$$\large
w_i = \left( \sum_{j \in \Psi_i} u_ju_j^T \right)^{-1} \sum_{j \in \Psi_i} r_{ij}u_j
$$
$$\large
u_j = \left( \sum_{i \in \Omega_j} w_iw_i^T \right)^{-1} \sum_{i \in \Omega_j} r_{ij}w_i
$$
gdzie:
- $\Psi_i$ oznacza zbi?r przedmiot?w, kt?re oceni? u?ytkownik $i$
- $\Omega_j$ oznacza zbi?r u?ytkownik?w, kt?rzy ocenili przedmiot $j$

Mo?na zauwa?y?, ?e w obu przypadkach jest to zwyczajny nadokre?lony (*overdetermined*) uk?ad r?wna? postaci $x=A^{-1}b$. Taki problem rozwi?zuje si? metod? najmniejszych kwadrat?w (*ordinary least squares*), st?d cz??? nazwy metody. Oczywi?cie do rozwi?zania problemu OLS mo?na u?y? SVD.

Przybli?ymy teraz kr?tko metod? ALS, bo SGD znamy ju? z sieci neuronowych. Mo?na zauwa?y? w powy?szych wzorach, ?e dla $W$ pochodna zale?y od warto?ci w $U$, a dla $U$ od warto?ci w $W$ - wydaje si?, ?e sytuacja patowa. Rozwi?zaniem jest po prostu przyj?? losowy punkt wyj?cia, a potem raz rozwi?zywa? $W$ za pomoc? $U$, a raz na odwr?t.

Pe?ny algorytm ALS:
1. Zainicjalizuj losowo macierze $W$ i $U$ niewielkimi warto?ciami z rozk?adu normalnego
2. Powtarzaj przez T krok?w:
  1. Zaktualizuj $U$ wed?ug wzoru, rozwi?zuj?c uk?ad r?wna?; $W$ jest sta?e
  2. Zaktualizuj $W$ wed?ug wzoru, rozwi?zuj?c uk?ad r?wna?; $U$ jest sta?e

Co wa?ne, zbie?no?? i ALS, i spadku wzd?u? gradientu jest gwarantowana, ale do minimum lokalnego. Zwykle nie stanowi to jednak problemu, a w razie czego zawsze mo?na wytrenowa? wiele modeli na r??nych `random_state` i wybra? najlepszy. Liczba epok treningowych stanowi do?? prosty hiperparametr - im wi?cej, tym dok?adniejsi po prostu b?dziemy, wi?c mo?emy bardziej overfitowa? (analogicznie do sieci neuronowych). Zazwyczaj w przypadku ALS wystarcza niewielka liczba, kilka-kilkana?cie.

Surprise implementuje wersj? z SGD. Wersj? z ALS implementuje np. Apache Spark. Wersj? z SGD mo?na te? ?atwo zaimplementowa? w dowolnym frameworku do sieci neuronowych, np. PyTorch czy TensorFlow.

In [ ]:
from surprise.prediction_algorithms.matrix_factorization import SVD

# regular MF - no user/item bias, no regularization
mf = SVD(biased=False, reg_all=0, random_state=0)
mf.fit(train_set)
pred_mf = mf.test(test_set)
rec_mf = get_recommendations(pred_mf)

print_metrics(pred_mf, rec_mf)

Dostali?my ca?kiem dobry wynik bez ?adnego tuningu, a czeka nas jeszcze troch? ulepsze?, bo powy?szy algorytm to jeszcze nie s?ynny FunkSVD.

Skoro user bias i item bias pomaga?y w metodach opartych o s?siedztwo, to dodajmy je te? tutaj, czemu nie. Najpierw mo?emy odj?? od wszystkiego globaln? ?redni? $\mu$, ?eby wycentrowa? ca?? macierz. Potem odejmujemy od ka?dego wiersza user bias $b_u$, a na koniec od ka?dej kolumny item bias $b_i$. Predykcja to zatem:
$$\large
\hat{r}_{ui} = \mu + b_u + b_i + w_u^Tu_i
$$

Drugie ulepszenie to dodanie regularyzacji do naszej funkcji kosztu. W ko?cu nie mo?emy si? zbyt bardzo dostosowa? do zbioru treningowego, nasz algorytm ma generalizowa? si? dla przysz?ych rekomendacji. Co wa?ne, mamy tutaj a? 4 mo?liwe ?r?d?a przeuczenia:
- $b_u$ - zbytnie dostosowanie do dotychczasowych odchyle? u?ytkownik?w
- $b_i$ - analogicznie, ale dla przedmiot?w
- $w_i$ - jest to wektor wag, wi?c du?e wagi oznaczaj? overfitting, jak np. w regresji liniowej
- $u_i$ - analogicznie, ale dla drugiej macierzy

Mo?na by u?y? 4 osobnych wsp??czynnik?w regularyzacji, ale optymalizacja takiej siatki hiperparametr?w jest raczej ma?o wykonalna. Mo?na wi?c u?y? jednego hiperparametru na moc regularyzacji L2 $\lambda$, w??czaj?c do niego wszystkie parametry. Daje to funkcj? kosztu:
$$\large
L = \sum_{u, i \in \Omega} \left( r_{ui} - \hat{r}_{ui} \right)^2 + \lambda \left( ||W||_2^2 + ||U||_2^2 + ||b_u||_2^2 + ||b_i||_2^2 \right)
$$

Pomijaj?c dalsze wyprowadzenie, nic nie zmienia to w gruncie rzeczy w algorytmie ALS, dalej mo?emy u?y? zwyk?ego OLS, zmieni? si? tylko troch? warto?ci w macierzach. Niewiele zmienia si? te?, gdy u?ywamy spadku wzd?u? gradientu - dodajemy tylko regularyzacj? do funkcji kosztu.

Powy?sze sformu?owanie to ju? pe?ny algorytm FunkSVD. Zobaczmy, jak sobie poradzi.

In [ ]:
funk_svd = SVD(biased=True, random_state=0)
funk_svd.fit(train_set)
pred_funk_svd = funk_svd.test(test_set)
rec_funk_svd = get_recommendations(pred_funk_svd)

print_metrics(pred_funk_svd, rec_funk_svd)

Wygl?da to na bardzo dobry wynik, a nie dokonali?my jeszcze ?adnego tuningu hiperparametr?w.

**Zadanie 9 (0.5 punktu)**

Zaimplementuj tuning hiperparametr?w dla algorytmu FunkSVD, sprawdzaj?c siatk? hiperparametr?w:
```
param_grid = {
    "n_factors": list(range(50, 151, 10)),
    "lr_all": [0.001, 0.003, 0.005, 0.007, 0.01],
    "reg_all": [0.01, 0.02, 0.03]
}
```

Pami?taj, aby przekaza? sta?e `random_state`! Przyda si? te? `n_jobs`. Je?eli na twoim sprz?cie b?dzie si? to liczy? o wiele za d?ugo, to mo?esz zmniejszy? zakres `n_factors` do 80-121.

Skomentuj wyniki.

In [ ]:
param_grid = {
    "n_factors": list(range(50, 151, 10)),
    "lr_all": [0.001, 0.003, 0.005, 0.007, 0.01],
    "reg_all": [0.01, 0.02, 0.03],
    "verbose": [True],
    "random_state": [0],
    "biased": [True]
}


model_grid = GridSearchCV(algo_class=SVD, param_grid=param_grid, measures=['fcp'], cv=10, n_jobs=-1, refit=True, joblib_verbose=3)
model_grid.fit(data_train)

In [ ]:
print(f"Best params {model_grid.best_params['fcp']}")
model = model_grid.best_estimator['fcp']
pred_knn_basic = model.test(test_set)
rec_knn_basic = get_recommendations(pred_knn_basic)

print_metrics(pred_knn_basic, rec_knn_basic)

Wyniki s? bardzo podobne jak w innych algorytmach. Jednak ma lepsze warto?ci metryk RMSE oraz MAE.

### Metody oparte o rozk?ad macierzy - podsumowanie

Podsumowanie:
1. Macierz ocen mo?na zdekomponowa? do iloczynu macierzy u?ytkownik?w $W$ oraz macierzy przedmiot?w $U$.
2. W tym podej?ciu wprowadzamy dodatkowy ukryty wymiar (latent dimension) wielko?ci $K$, kt?ry reprezentuje tematy ukryte w naszych danych.
3. Do obliczania macierzy minimalizuje si? b??d przybli?enia macierzy ocen przez nasze macierze $W$ i $U$. S?u?y do tego albo spadek wzd?u? gradientu, albo, bardziej typowo, algorytm Alternating Least Squares (ALS).

Zalety:
1. Bardzo dobre wyniki
2. Szybko?? i skalowalno??
3. Mo?liwo?? przyspieszenia oblicze? z pomoc? GPU
4. Dzia?a do?? dobrze w przypadku zimnego startu.

Wady:
1. Do?? du?o hiperparametr?w, przynajmniej w przypadku u?ycia spadku wzd?u? gradientu
2. Brak optymalnego wyniku, trzeba by wypr?bowa? r??ne losowe punkty startowe
3. Niska interpretowalno??


## Projektowanie system?w rekomendacyjnych

Podczas projektowania faktycznych system?w rekomendacyjnych trzeba rozwa?y? wiele nieoczywistych kwestii. O wielu z nich m?wili?my ju? podczas tego laboratorium. Teraz mo?na zmierzy? si? z paroma takimi kwestiami.

**Zadanie 10 (1 punkt)**

Odpowiedz na poni?sze pytania. Ka?de pytanie jest warte 0.25 punktu.

1. Za???my, ?e mamy system rekomenduj?cy reklamy u?ytkownikom. Reklamy zmieniaj? si? regularnie i do?? cz?sto. Czy algorytmy collaborative filtering b?d? tu dobrym wyborem, a je?li tak, to jakie? Je?eli nie, to co b?dzie stanowi? tu g??wny problem?

2. Wymy?l i kr?tko opisz architektur? przyk?adowego systemu rekomendacyjnego, sugeruj?cego gry w sklepie Steam. Opisz, jakich algorytm?w u?y?by? w konkretnych przypadkach i czemu. Uwzgl?dnij, ?e mo?e wyst?pi? zimny start w przypadku nowych u?ytkownik?w oraz gier, oraz ?e mamy ogromn? przestrze? mo?liwych gier - ka?dy u?ytkownik zagra tylko w niewielki u?amek.

3. Pracujesz w firmie obs?uguj?cej platformy z newsami jako data scientist. Zesp?? data engineering zgromadzi? bardzo du?? ilo?? log?w o ludziach klikaj?cych w artyku?y i linki (clickstream data). U?ytkownicy nie musz? logowa? si? do systemu, wi?c identyfikacja jest oparta o ciasteczka (cookies) i niepewna pomi?dzy sesjami. Posiadasz za to bogate metadane o u?ytkowniku (np. rodzaj urz?dzenia, przegl?darki, geolokacja, historia sesji) oraz o artyku?ach (np. j?zyk, tre??, tagi). Jakiego rodzaju systemu rekomendacyjnego by? u?y? (jakiego da si? u?y??) i dlaczego? Nie musisz tutaj opisywa? szczeg??owo algorytm?w. Rozwa?:
  - globalny vs personalizowany
  - content-based vs collaborative filtering vs hybrydowy
  - explicit vs implicit

4. Pracujesz w firmie tworz?cej oprogramowanie dla sklep?w internetowych. U?ytkownik ma mo?liwo?? sortowania malej?co po ?redniej opinii, ale z historii u?y? wynika, ?e cz?sto nie klikaj? przedmiot?w o najwy?szej ?redniej ocenie, tylko kt?ry? z kolejnych. Jaki mo?e by? tego przyczyna? Jak mo?na by rozwi?za? ten problem?

1. Mog? si? okaza? nieskuteczne ze wzgl?du na to, ?e itemy si? szybko zmieniaj? i nie b?dziemy w stanie korzysta? z historii ocen u?ytkownik?w. Na pewno mieliby?my do czynienia z zimnym startem.

2. Dla nowych u?ytkownik?w mogliby?my u?ywa? globalnych metod rekomendacji i proponowa? tytu?y, kt?re s? w danej chwili najbardziej popularne. Nast?pnie po jakim? okresie czasu, gdy u?tkownik mia?by ju? jak?? liczb? wystawionych ocen lub godzin sp?dzonych w danych grach mo?na by u?y? metod spersonalizowanych na przyk?ad user-based KNN.

3. Skoro mamy du?? ilo?? informacji o u?ytkowniku mo?na u?y? systemy personalizowanego. Lepsze w tym wypadku b?d? podej?cia implicit, gdy? nasze dane nie s? to oceny, kt?re u?ytkownik daje wprost i mo?na by u?y? podej?cia hybrydowego.

4. U?ytkownicy ch?tniej b?d? klika? w przedmioty kt?re poza wysok? ?redni? b?d? mia?y tak?e du?? liczb? ocen. Mo?na by doda? pole, kt?re sortowa?o by produkty po warto?ci ?redniej bayesowskiej.


?r?d?a inspiracji do zada? dla zainteresowanych:
1. Praca autora tego zestawu.
2. Steam API oraz SteamSpy pozwalaj? ?atwo zbiera? du?e zbiory danych z tej platformy, powsta?o w ten spos?b zreszt? ca?kiem sporo projekt?w, prac dyplomowych i artyku??w naukowych (np. [projekt 1](http://brandonlin.com/steam.pdf), [projekt 2](https://library.ucsd.edu/dc/object/bb5021836n/_3_1.pdf), [praca dyplomowa](https://openaccess.mef.edu.tr/bitstream/handle/20.500.11779/1721/Serhan%20Bayram.pdf?sequence=1&isAllowed=y), [artyku? naukowy 1](https://www.researchgate.net/publication/333072035_Recommender_Systems_for_Online_Video_Game_Platforms_the_Case_of_STEAM), [artyku? naukowy 2](https://trepo.tuni.fi/bitstream/handle/10024/122499/a_hybrid_recommender_system_2020.pdf;jsessionid=E796B8E915FBBF37EF1E0B75210D8690?sequence=2)). Przyk?adowe zbiory: [dataset 1](https://www.kaggle.com/datasets/nikdavis/steam-store-games), [dataset 2](https://www.kaggle.com/datasets/forgemaster/steam-reviews-dataset), [dataset 3](https://cseweb.ucsd.edu/~jmcauley/datasets.html#steam_data).
3. Lu?no wzorowane na [zbiorze danych CI&T Deskdrop](https://www.kaggle.com/datasets/gspmoreira/articles-sharing-reading-from-cit-deskdrop).
4. Artyku? ["How not to sort by average rating"](https://www.evanmiller.org/how-not-to-sort-by-average-rating.html).

## Zadanie dla ch?tnych

Uruchom na zbiorze MovieLens-1M (albo innym podobnego rozmiaru) algorytm LightGCN ([artyku?](https://arxiv.org/pdf/2002.02126.pdf)), implementuj?cy podej?cie grafowe do rekomendacji, z u?yciem biblioteki LibRecommender ([tutorial](https://github.com/massquantity/LibRecommender/blob/master/examples/pure_ranking_example.py)), kt?ra pod spodem u?ywa PyTorch Geometric ([tutorial dla odwa?nych](https://colab.research.google.com/drive/1VfP6JlWbX_AJnx88yN1tM3BYE6XAADiy?usp=sharing)). Poni?ej opis, jak to dzia?a, ale znajomo?? teorii nie jest potrzebna do wykonania tego zadania :) Mo?esz u?y? domy?lnych hiperparametr?w architektury sieci z tutoriala, ale zaimplementuj tuning przynajmniej liczby epok (LibRecommender nie ma early stoppingu). Je?eli zbi?r 1M jest za du?y dla twojego sprz?tu, mo?esz pracowa? na 100k. Je?eli u?yjesz MovieLens-1M, dla por?wnania zaimplementuj tak?e wybrane 1-2 algorytmy z tego laboratorium na tym zbiorze (mo?esz po prostu skopiowa? kod z notebooka powy?ej).

W tym podej?ciu reprezentujemy problem jako graf, a nie jako macierz. Mamy graf dwudzielny u?ytkownik?w i przedmiot?w, gdzie ocena reprezentowana jest jako kraw?d? mi?dzy wierzcho?kiem u?ytkownika a wierzcho?kiem przedmiotu, opisana ocen?. Rekomendacja polega na zadaniu **przewidywania kraw?dzi (edge prediction)**, czyli zasugerowanie dodania nowej kraw?dzi mi?dzy u?ytkownikiem a przedmiotem.

Sie? LightGCN implementuje podej?cie collaborative filtering na grafie. Jest to **grafowa sie? neuronowa (Graph Neural Network, GNN)**, osi?gaj?ca obecnie jedne z najlepszych wynik?w w?r?d system?w CF. Ka?dy wierzcho?ek ma tutaj wektor o pewnej za?o?onej z g?ry d?ugo?ci $N$, tzw. embedding. Tworzy si? go nast?puj?co:
- robimy one-hot encoding dla u?ytkownik?w i przedmiot?w, koduj?c ich `user_id` i `item_id`
- mno?ymy u?ytkownik?w przez macierz, robi?c kombinacj? liniow? i rzutuj?c na ni?szy wymiar
- to samo, co wy?ej, tylko dla przedmiot?w
Macierze embedduj?ce dla u?ytkownik?w i przedmiot?w s? parametrami, kt?rych uczymy si? wraz z treningiem sieci neuronowej. Inicjalizuje si? je losowo.

Sie? LightGCN sk?ada si? z kilku warstw **konwolucji grafowej (graph convolution)**, gdzie ka?da warstwa agreguje informacj? z s?siednich wierzcho?k?w. Dla ka?dego wierzcho?ka robimy po prostu sum? wa?on? wektor?w s?siad?w ($e_u$ - embedding u?ytkownika $u$, $e_i$ - embedding przedmiotu $i$):
$$\large
e_u^{(k+1)} = \sum_{i \in N(u)} \frac{1}{\sqrt{N(u)}\sqrt{N(i)}} e_i^{(k)}
$$
S?siad?w wa?ymy ich stopniem, aby wzi?? pod uwag? popularno?? poszczeg?lnych przedmiot?w i aktywno?? u?ytkownik?w (maj? du?y stopie?). Taka wymiana informacji mi?dzy wierzcho?kami propaguje informacj? w grafie, aktualizuj?c embeddingi.

Typowo takich warstw jest kilka, np. 3-4. P??niej agreguje si? informacj? ze wszystkich warstw, w odr??nieniu od sieci CNN dla obraz?w, gdzie zwykle bierze si? wyj?cie tylko z ostatniej warstwy. Dla ka?dego u?ytkownika (i przedmiotu) bierzemy jego embedding z ka?dej warstwy i u?redniamy je. Daje to bogat? reprezentacj? wierzcho?ka i agreguje informacje zar?wno z bliskiego s?siedztwa (g??bokie warstwy), jak i z og??u spo?eczn?ci w grafie (wysokie warstwy).

Predykcja to po prostu iloczyn skalarny embeddingu u?ytkownika i przedmiotu: $r_{ij} = e_j^T e_i$. Sie? tak? uczy si? zwykle funkcj? kosztu **Bayesian Personalized Ranking (BPR)**, u?ywan? powszechnie w sieciach neuronowych do system?w rekomendacyjnych. Oczywi?cie uwzgl?dnia si? tu wszystkie typowe elementy sieci neuronowych: learning rate, weight decay etc.